<a href="https://colab.research.google.com/github/amita-kapoor/Agentic-Systems-Engineering/blob/main/Chapter05/ch05_action_engine_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 5: The Action Engine — Code Review Notebook

This notebook contains every code listing from Chapter 5 of *Agentic Systems Engineering*,  
assembled in dependency order and integrated with the Anthropic Claude API so readers  
can run the complete end-to-end pipeline.

**How to use this notebook:**
1. Set your `ANTHROPIC_API_KEY` environment variable, or paste it in Cell 2
2. Run cells top-to-bottom — later cells depend on earlier definitions
3. Section 9 runs the complete end-to-end demo with a live Claude model



## Cell 1 — Install dependencies

In [ ]:
%pip install anthropic pydantic sentence-transformers numpy --quiet
# Optional for Listing 5.19:
%pip install playwright --quiet
!playwright install chromium --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.8/478.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 19.4 MB/s eta 0:00:00
error: unknown option '--quiet'


## Cell 2 — API key

In [ ]:
import os
from google.colab import userdata

# Read the Anthropic API key from Colab Secrets securely
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except userdata.SecretNotFoundError:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY is not set. "
        "Please add it to your Colab Secrets (the key icon on the left sidebar)."
    )

import anthropic
client = anthropic.Anthropic()
print("Anthropic client ready.")

Anthropic client ready.


---
## Listing 5.1: Structured interface with outcome states



In [ ]:
# Listing 5.1 — Structured interface for a customer lookup action
from enum import Enum
from typing import Optional

class Status(Enum):
    OK = "ok"
    NOT_FOUND = "not_found"
    AMBIGUOUS = "ambiguous"

class CustomerLookupResult:
    def __init__(self, status, customer=None, candidates=None):
        self.status = status
        self.customer = customer
        self.candidates = candidates

def get_customer_v0(customer_id: str) -> CustomerLookupResult:
    """Minimal demonstration of structured outcomes."""
    database = {
        "cust_1": {"name": "Asha Gupta", "email": "asha@example.com"},
    }
    record = database.get(customer_id)
    if record is None:
        return CustomerLookupResult(status=Status.NOT_FOUND)
    return CustomerLookupResult(status=Status.OK, customer=record)

# Test: caller branches on outcome, not on raw data
result = get_customer_v0("cust_1")
if result.status == Status.OK:
    print(f"Found: {result.customer}")
elif result.status == Status.NOT_FOUND:
    print("Not found — would ask user for new identifier")

result2 = get_customer_v0("cust_999")
print(f"Missing customer status: {result2.status}")

Found: {'name': 'Asha Gupta', 'email': 'asha@example.com'}
Missing customer status: Status.NOT_FOUND


---
## Listing 5.2: Minimal typed tool contract


In [ ]:
# Listing 5.2 — Minimal typed tool with structured inputs and outputs
from pydantic import BaseModel
from enum import Enum
from typing import Optional

class ToolStatus(str, Enum):          # renamed to avoid clash with Listing 5.1
    SUCCESS = "success"
    NOT_FOUND = "not_found"
    RETRYABLE_ERROR = "retryable_error"

class CustomerLookupInput(BaseModel):
    customer_id: str

class CustomerRecord(BaseModel):
    customer_id: str
    name: str
    email: str

class CustomerLookupResultV0(BaseModel):
    status: ToolStatus
    customer: Optional[CustomerRecord] = None
    error_message: Optional[str] = None

def get_customer_typed(input: CustomerLookupInput) -> CustomerLookupResultV0:
    """
    Tool name: get_customer
    Description: Retrieve a customer record by unique identifier.
    """
    database = {
        "cust_1": CustomerRecord(
            customer_id="cust_1",
            name="Asha Gupta",
            email="asha@example.com"
        )
    }
    record = database.get(input.customer_id)
    if record is None:
        return CustomerLookupResultV0(status=ToolStatus.NOT_FOUND)
    return CustomerLookupResultV0(status=ToolStatus.SUCCESS, customer=record)

# Test
inp = CustomerLookupInput(customer_id="cust_1")
res = get_customer_typed(inp)
print(f"Status: {res.status}")
print(f"Customer: {res.customer}")

# Validation test — Pydantic catches bad input
try:
    bad = CustomerLookupInput(customer_id=123)   # should coerce or raise
    print(f"Coerced to str: {bad.customer_id!r}")
except Exception as e:
    print(f"Validation error: {e}")

Status: ToolStatus.SUCCESS
Customer: customer_id='cust_1' name='Asha Gupta' email='asha@example.com'
Validation error: 1 validation error for CustomerLookupInput
customer_id
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


---
## Listing 5.3: MCP JSON Schema (reference)



In [ ]:
# Listing 5.3 — MCP JSON Schema (reference — not executed)
import json

mcp_tool_schema = {
    "name": "get_customer",
    "description": "Retrieve a customer record by unique identifier.",
    "inputSchema": {
        "type": "object",
        "title": "Get Customer Input",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "Unique identifier for the customer"
            }
        },
        "required": ["customer_id"],
        "additionalProperties": False
    }
}

print("MCP tool schema:")
print(json.dumps(mcp_tool_schema, indent=2))

print("\nNote: This is the same contract as the Python class above, expressed")
print("as JSON Schema for the MCP wire protocol.")

MCP tool schema:
{
  "name": "get_customer",
  "description": "Retrieve a customer record by unique identifier.",
  "inputSchema": {
    "type": "object",
    "title": "Get Customer Input",
    "properties": {
      "customer_id": {
        "type": "string",
        "description": "Unique identifier for the customer"
      }
    },
    "required": [
      "customer_id"
    ],
    "additionalProperties": false
  }
}

Note: This is the same contract as the Python class above, expressed
as JSON Schema for the MCP wire protocol.


---
## Listing 5.4 & 5.5: Tool metadata, ActionResult, and policy enforcement



In [ ]:
# Listings 5.4 + 5.5 — Full metadata contract with runtime enforcement
from pydantic import BaseModel, Field, model_validator
from enum import Enum
from typing import Optional, Type, Any, Literal
import asyncio
import time


# --- Risk classification ---
class RiskLevel(str, Enum):
    LOW      = "low"       # Read-only, no side effects — safe to retry freely
    MEDIUM   = "medium"    # Reversible side effects — retry with idempotency key
    HIGH     = "high"      # Irreversible side effects — log, alert, consider HITL
    CRITICAL = "critical"  # Financial, legal, or safety impact — HITL mandatory


# --- Domain result: what the business logic found ---
class CustomerLookupOutput(BaseModel):
    """Responsibility: Business outcome."""
    status: Literal["found", "not_found"]
    customer: Optional[dict] = None


# --- Execution envelope: how the call went ---
class ActionResult(BaseModel):
    """Responsibility: What happened during execution."""
    status: Literal["success", "failure", "partial", "pending"]
    output: Optional[Any] = None
    error: Optional[dict] = None
    latency_ms: float = 0.0


# --- Tool metadata: how the tool should be treated ---
class ToolMetadata(BaseModel):
    """Responsibility: How the tool should be treated."""
    name: str
    description: str
    version: str = "1.0.0"           # Added in section 5.2.4
    args_schema: Type[BaseModel]
    risk_level: RiskLevel = RiskLevel.LOW
    is_idempotent: bool = True
    timeout_seconds: float = 5.0
    cost_estimate_usd: Optional[float] = None
    requires_confirmation: bool = False
    preconditions: list[str] = Field(default_factory=list)
    postconditions: list[str] = Field(default_factory=list)

    @model_validator(mode="after")
    def enforce_confirmation_for_critical(self) -> "ToolMetadata":
        if self.risk_level == RiskLevel.CRITICAL and not self.requires_confirmation:
            raise ValueError(
                f"Tool '{self.name}' is CRITICAL risk but requires_confirmation=False. "
                "CRITICAL tools must require human confirmation."
            )
        return self


# --- Tool base class ---
class BaseTool:
    metadata: ToolMetadata

    async def execute(self, input: BaseModel) -> ActionResult:
        start = time.monotonic()
        try:
            result = await asyncio.wait_for(
                self._run(input),
                timeout=self.metadata.timeout_seconds,
            )
            return ActionResult(
                status="success",
                output=result,
                latency_ms=(time.monotonic() - start) * 1000,
            )
        except asyncio.TimeoutError:
            return ActionResult(
                status="failure",
                error={
                    "code": "TimeoutError",
                    "message": "Tool execution timed out",
                    "retryable": True,
                },
                latency_ms=self.metadata.timeout_seconds * 1000,
            )

    async def _run(self, input: BaseModel) -> Any:
        raise NotImplementedError


# --- Customer lookup tool (Listing 5.4) ---
class CustomerLookupTool(BaseTool):
    metadata = ToolMetadata(
        name="get_customer",
        description="Retrieve a customer record by unique identifier.",
        args_schema=CustomerLookupInput,
        risk_level=RiskLevel.LOW,
        is_idempotent=True,
        timeout_seconds=2.0,
        requires_confirmation=False,
        postconditions=["customer record available in context"],
    )

    async def _run(self, input: CustomerLookupInput) -> CustomerLookupOutput:
        # In production: record = await db.get(input.customer_id)
        database = {"cust_1": {"name": "Asha Gupta", "email": "asha@example.com"}}
        record = database.get(input.customer_id)
        if record is None:
            return CustomerLookupOutput(status="not_found")
        return CustomerLookupOutput(status="found", customer=record)


# --- Test Listing 5.4 ---
async def test_listing_54():
    tool = CustomerLookupTool()
    inp = CustomerLookupInput(customer_id="cust_1")
    result = await tool.execute(inp)
    print(f"Execution status : {result.status}")
    print(f"Domain outcome   : {result.output.status}")
    print(f"Customer         : {result.output.customer}")
    print(f"Latency          : {result.latency_ms:.1f}ms")

    # Not-found is still execution success
    result2 = await tool.execute(CustomerLookupInput(customer_id="cust_999"))
    print(f"\nMissing customer — execution: {result2.status}, domain: {result2.output.status}")

await test_listing_54()

Execution status : success
Domain outcome   : found
Customer         : {'name': 'Asha Gupta', 'email': 'asha@example.com'}
Latency          : 0.2ms

Missing customer — execution: success, domain: not_found


In [ ]:
# Listing 5.5 — CRITICAL policy enforcement fires at definition time

class WireTransferInput(BaseModel):
    account_number: str
    amount_usd: float
    recipient_name: str

print("Attempting to define a CRITICAL tool without requires_confirmation...")
try:
    class InitiateWireTransferTool_BAD(BaseTool):
        metadata = ToolMetadata(
            name="initiate_wire_transfer",
            description="Transfer funds to an external bank account.",
            args_schema=WireTransferInput,
            risk_level=RiskLevel.CRITICAL,
            requires_confirmation=False,   # ← violates policy
        )
except ValueError as e:
    print(f"  ValueError caught at definition time: {e}")

print("\nDefining the CORRECT version with requires_confirmation=True...")
class InitiateWireTransferTool(BaseTool):
    metadata = ToolMetadata(
        name="initiate_wire_transfer",
        description="Transfer funds to an external bank account.",
        args_schema=WireTransferInput,
        risk_level=RiskLevel.CRITICAL,
        requires_confirmation=True,
        is_idempotent=False,
        timeout_seconds=10.0,
        postconditions=["transfer initiated", "confirmation reference stored"],
    )

    async def _run(self, input: WireTransferInput) -> dict:
        # Real implementation would call payment API
        return {"transfer_id": "txn_abc123", "status": "initiated"}

print(f"  Wire transfer tool created successfully.")
print(f"  Risk level: {InitiateWireTransferTool.metadata.risk_level}")
print(f"  Requires confirmation: {InitiateWireTransferTool.metadata.requires_confirmation}")

Attempting to define a CRITICAL tool without requires_confirmation...
  ValueError caught at definition time: 1 validation error for ToolMetadata
  Value error, Tool 'initiate_wire_transfer' is CRITICAL risk but requires_confirmation=False. CRITICAL tools must require human confirmation. [type=value_error, input_value={'name': 'initiate_wire_t...es_confirmation': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

Defining the CORRECT version with requires_confirmation=True...
  Wire transfer tool created successfully.
  Risk level: RiskLevel.CRITICAL
  Requires confirmation: True


---
## Listings 5.6 & 5.7: ToolRegistry with semantic search and risk filtering

In [ ]:
# Listing 5.6 — ToolRegistry with semantic search
# Listing 5.7 — search() extended with risk filtering (replaces Listing 5.6 search)
import numpy as np
from sentence_transformers import SentenceTransformer

class ToolRegistry:
    def __init__(self):
        self._tools: dict[str, BaseTool] = {}
        self._tool_names: list[str] = []
        self._embeddings: Optional[np.ndarray] = None
        self._model = SentenceTransformer("all-MiniLM-L6-v2")

    def register(self, tool: BaseTool) -> None:
        """Add a new tool and store its description embedding."""
        name = tool.metadata.name
        if name in self._tools:
            raise ValueError(
                f"Tool '{name}' is already registered. Use update() to replace it."
            )
        self._tools[name] = tool
        self._tool_names.append(name)
        new_emb = self._model.encode(
            [tool.metadata.description],
            normalize_embeddings=True,
        )
        self._embeddings = (
            new_emb
            if self._embeddings is None
            else np.vstack([self._embeddings, new_emb])
        )

    def get(self, name: str) -> Optional[BaseTool]:
        """Look up a tool by name."""
        return self._tools.get(name)

    def all_tools(self) -> list[BaseTool]:
        """Return all registered tools (for passing to model)."""
        return list(self._tools.values())

    def search(
        self,
        query: str,
        top_k: int = 3,
        max_risk: RiskLevel = RiskLevel.CRITICAL,  # Listing 5.7 extension
    ) -> list[BaseTool]:
        """Return relevant tools after filtering by risk level.

        max_risk=CRITICAL (default) imposes no restriction — all tools are eligible.
        Pass max_risk=MEDIUM to exclude HIGH and CRITICAL tools from results.
        """
        if not self._tools:
            return []

        # Step 1 (Listing 5.7): filter tools based on risk level
        risk_order = list(RiskLevel)
        eligible = [
            (i, name)
            for i, name in enumerate(self._tool_names)
            if risk_order.index(self._tools[name].metadata.risk_level)
               <= risk_order.index(max_risk)
        ]
        if not eligible:
            return []

        # Step 2: perform semantic search on filtered tools
        indices, names = zip(*eligible)
        eligible_embs = self._embeddings[list(indices)]
        query_emb = self._model.encode([query], normalize_embeddings=True)[0]
        scores = np.dot(eligible_embs, query_emb)
        effective_k = min(top_k, len(eligible))
        top_local = np.argsort(scores)[-effective_k:][::-1]
        return [self._tools[names[i]] for i in top_local]


# --- Test the registry ---
registry = ToolRegistry()

# Register a few tools with different risk levels
class RefundInput(BaseModel):
    order_id: str
    amount_usd: float

class IssuedRefundTool(BaseTool):
    metadata = ToolMetadata(
        name="issue_refund",
        description="Issue a refund for a customer order. Modifies billing records.",
        args_schema=RefundInput,
        risk_level=RiskLevel.HIGH,
        is_idempotent=True,
        requires_confirmation=True,
    )
    async def _run(self, input): return {"refund_id": "ref_001"}

class SearchOrdersInput(BaseModel):
    customer_id: str

class SearchOrdersTool(BaseTool):
    metadata = ToolMetadata(
        name="search_orders",
        description="Search order history for a customer account.",
        args_schema=SearchOrdersInput,
        risk_level=RiskLevel.LOW,
    )
    async def _run(self, input): return [{"order_id": "ord_1"}]

registry.register(CustomerLookupTool())
registry.register(IssuedRefundTool())
registry.register(SearchOrdersTool())
registry.register(InitiateWireTransferTool())

print("=== Listing 5.6: semantic search (no risk filter) ===")
results = registry.search("look up customer account information", top_k=3)
for t in results:
    print(f"  {t.metadata.name} [{t.metadata.risk_level}]")

print("\n=== Listing 5.7: risk-filtered search (max_risk=MEDIUM) ===")
safe_results = registry.search(
    "look up customer account information",
    top_k=3,
    max_risk=RiskLevel.MEDIUM,
)
for t in safe_results:
    print(f"  {t.metadata.name} [{t.metadata.risk_level}]")
print("  (issue_refund and initiate_wire_transfer correctly excluded)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

=== Listing 5.6: semantic search (no risk filter) ===
  search_orders [RiskLevel.LOW]
  get_customer [RiskLevel.LOW]
  issue_refund [RiskLevel.HIGH]

=== Listing 5.7: risk-filtered search (max_risk=MEDIUM) ===
  search_orders [RiskLevel.LOW]
  get_customer [RiskLevel.LOW]
  (issue_refund and initiate_wire_transfer correctly excluded)


---
## Listings 5.8 & 5.9: Tool versioning and the adapter pattern

In [ ]:
# Listings 5.8 + 5.9 — Versioned tool contracts and V1→V2 adapter
from typing import Literal, cast

# V1 contract
class CustomerLookupInputV1(BaseModel):
    customer_id: str

class CustomerLookupResultV1(BaseModel):
    status: Literal["found", "not_found"]
    customer: Optional[dict] = None

# V2 contract — adds include_orders parameter
class CustomerLookupInputV2(BaseModel):
    customer_id: str
    include_orders: bool = False

class CustomerLookupResultV2(BaseModel):
    status: Literal["found", "not_found"]
    customer: Optional[dict] = None
    recent_orders: list[dict] = []

# Distinct names prevent ToolRegistry collision (bug fix from review)
class CustomerLookupToolV1(BaseTool):
    metadata = ToolMetadata(
        name="get_customer",
        description="Retrieve a customer record by unique identifier.",
        args_schema=CustomerLookupInputV1,
        version="1.0.0",
    )
    async def _run(self, input: CustomerLookupInputV1) -> CustomerLookupResultV1:
        database = {"cust_1": {"name": "Asha Gupta", "email": "asha@example.com"}}
        record = database.get(input.customer_id)
        if record is None:
            return CustomerLookupResultV1(status="not_found")
        return CustomerLookupResultV1(status="found", customer=record)

class CustomerLookupToolV2(BaseTool):
    metadata = ToolMetadata(
        name="get_customer",
        description="Retrieve a customer record and optionally include recent orders.",
        args_schema=CustomerLookupInputV2,
        version="2.0.0",
    )
    async def _run(self, input: CustomerLookupInputV2) -> CustomerLookupResultV2:
        database = {"cust_1": {"name": "Asha Gupta", "email": "asha@example.com"}}
        record = database.get(input.customer_id)
        if record is None:
            return CustomerLookupResultV2(status="not_found")
        orders = [{"order_id": "ord_1", "amount": 49.99}] if input.include_orders else []
        return CustomerLookupResultV2(status="found", customer=record, recent_orders=orders)


# Listing 5.9 — V1 adapter wrapping V2 implementation
class CustomerLookupAdapterV1(BaseTool):
    metadata = ToolMetadata(
        name="get_customer_v1_compat",
        description="Compatibility wrapper for version 1 callers.",
        args_schema=CustomerLookupInputV1,
        version="1.0.0",
    )

    def __init__(self, v2_tool: CustomerLookupToolV2):
        self.v2_tool = v2_tool

    async def execute(self, input: CustomerLookupInputV1) -> ActionResult:
        v2_input = CustomerLookupInputV2(
            customer_id=input.customer_id,
            include_orders=False,
        )
        result = await self.v2_tool.execute(v2_input)
        if result.status != "success":
            return result
        v2_output = cast(CustomerLookupResultV2, result.output)
        v1_output = CustomerLookupResultV1(
            status=v2_output.status,
            customer=v2_output.customer,
        )
        return ActionResult(
            status="success",
            output=v1_output,
            latency_ms=result.latency_ms,
        )

    async def _run(self, input): pass  # unused — execute() is overridden


# Test versioning
async def test_versioning():
    v2 = CustomerLookupToolV2()
    adapter = CustomerLookupAdapterV1(v2_tool=v2)

    print("=== V2 direct (with orders) ===")
    r = await v2.execute(CustomerLookupInputV2(customer_id="cust_1", include_orders=True))
    print(f"  status: {r.status}, orders: {r.output.recent_orders}")

    print("\n=== V1 adapter (no orders — V1 callers unaffected) ===")
    r2 = await adapter.execute(CustomerLookupInputV1(customer_id="cust_1"))
    print(f"  status: {r2.status}, output type: {type(r2.output).__name__}")
    print(f"  V1 output has no recent_orders field: {not hasattr(r2.output, 'recent_orders')}")

await test_versioning()

=== V2 direct (with orders) ===
  status: success, orders: [{'order_id': 'ord_1', 'amount': 49.99}]

=== V1 adapter (no orders — V1 callers unaffected) ===
  status: success, output type: CustomerLookupResultV1
  V1 output has no recent_orders field: True


---
## Listings 5.10–5.13: Safe execution primitives

In [ ]:
# Listing 5.10 — Deterministic idempotency key
import hashlib
import json

def make_idempotency_key(tool_name: str, inputs: dict) -> str:
    canonical = json.dumps(
        {"tool": tool_name, "inputs": inputs},
        sort_keys=True
    )
    return hashlib.sha256(canonical.encode()).hexdigest()  # note: no extra ) here

# Test — same inputs always produce same key
key1 = make_idempotency_key("get_customer", {"customer_id": "cust_1"})
key2 = make_idempotency_key("get_customer", {"customer_id": "cust_1"})
key3 = make_idempotency_key("get_customer", {"customer_id": "cust_2"})

print(f"Key 1: {key1[:16]}...")
print(f"Key 2: {key2[:16]}...  (identical — safe to retry)")
print(f"Key 3: {key3[:16]}...  (different customer — different key)")
assert key1 == key2
assert key1 != key3
print("Assertions passed.")

Key 1: 10460f6909603f70...
Key 2: 10460f6909603f70...  (identical — safe to retry)
Key 3: 7cb4613854d9e7cb...  (different customer — different key)
Assertions passed.


In [ ]:
# Listing 5.12 — CircuitBreaker
# (Listing 5.11 is superseded by Listing 5.15 below — shown here for reference)

class CircuitBreaker:
    """Three-state circuit breaker: CLOSED → OPEN → HALF_OPEN → CLOSED."""

    def __init__(self, failure_threshold: int = 5, reset_timeout_s: float = 60):
        self.failure_threshold = failure_threshold
        self.reset_timeout = reset_timeout_s
        self.failures = 0
        self.state = "CLOSED"
        self.last_failure_time = 0.0

    async def call(self, tool_fn, *args, **kwargs):
        if self.state == "OPEN":
            if time.monotonic() - self.last_failure_time > self.reset_timeout:
                self.state = "HALF_OPEN"
            else:
                raise RuntimeError("Circuit open — dependency unavailable")
        try:
            result = await tool_fn(*args, **kwargs)
            if self.state == "HALF_OPEN":
                self.state = "CLOSED"
                self.failures = 0
            return result
        except Exception:
            self.failures += 1
            self.last_failure_time = time.monotonic()
            if self.failures >= self.failure_threshold:
                self.state = "OPEN"
            raise


# Test the circuit breaker
async def test_circuit_breaker():
    breaker = CircuitBreaker(failure_threshold=2, reset_timeout_s=0.1)

    call_count = 0
    async def failing_tool():
        nonlocal call_count
        call_count += 1
        raise ConnectionError("Simulated failure")

    print(f"Initial state: {breaker.state}")

    for i in range(3):
        try:
            await breaker.call(failing_tool)
        except Exception as e:
            print(f"  Attempt {i+1}: {type(e).__name__} — breaker state: {breaker.state}")

    print(f"Circuit is now OPEN after {call_count} calls")
    print(f"Waiting for cooldown...")
    await asyncio.sleep(0.15)

    # After cooldown, breaker transitions to HALF_OPEN
    print(f"State before probe: {breaker.state}")
    try:
        await breaker.call(failing_tool)
    except RuntimeError as e:
        print(f"  Still blocked: {e}")
    except ConnectionError:
        print(f"  Probe attempt reached tool (HALF_OPEN), failed again → back to OPEN")
        print(f"  State: {breaker.state}")

await test_circuit_breaker()

Initial state: CLOSED
  Attempt 1: ConnectionError — breaker state: CLOSED
  Attempt 2: ConnectionError — breaker state: OPEN
  Attempt 3: RuntimeError — breaker state: OPEN
Circuit is now OPEN after 2 calls
Waiting for cooldown...
State before probe: OPEN
  Probe attempt reached tool (HALF_OPEN), failed again → back to OPEN
  State: OPEN


In [ ]:
# Listing 5.13 — AgentSaga with compensation stack

class AgentSaga:
    """Saga pattern: pair each step with a compensating action for rollback."""

    def __init__(self):
        self.compensations = []

    async def execute_step(self, action, compensate):
        result = await action()
        self.compensations.append(compensate)
        return result

    async def rollback(self):
        while self.compensations:
            compensate = self.compensations.pop()
            await compensate()


# Test: booking workflow with mid-saga failure
async def test_saga():
    log = []

    async def reserve():       log.append("RESERVED");   return {"id": "res_1"}
    async def cancel_reserve(): log.append("CANCELLED RESERVATION")

    async def confirm():       log.append("CONFIRMED");  return {"id": "conf_1"}
    async def undo_confirm():  log.append("UNDID CONFIRMATION")

    async def notify():        raise RuntimeError("Notification service down")

    print("=== Saga with failure at step 3 ===")
    saga = AgentSaga()
    try:
        await saga.execute_step(reserve, cancel_reserve)
        await saga.execute_step(confirm, undo_confirm)
        await saga.execute_step(notify, lambda: None)   # fails here
    except RuntimeError as e:
        print(f"  Step failed: {e}")
        await saga.rollback()

    print(f"  Execution log: {log}")
    assert "UNDID CONFIRMATION" in log
    assert "CANCELLED RESERVATION" in log
    print("  Rollback verified: compensation ran in reverse order.")

    print("\n=== Saga with full success ===")
    log.clear()
    saga2 = AgentSaga()
    async def notify_ok(): log.append("NOTIFIED")
    await saga2.execute_step(reserve, cancel_reserve)
    await saga2.execute_step(confirm, undo_confirm)
    await saga2.execute_step(notify_ok, lambda: None)
    print(f"  Execution log: {log}")
    print("  No rollback needed.")

await test_saga()

=== Saga with failure at step 3 ===
  Step failed: Notification service down
  Execution log: ['RESERVED', 'CONFIRMED', 'UNDID CONFIRMATION', 'CANCELLED RESERVATION']
  Rollback verified: compensation ran in reverse order.

=== Saga with full success ===
  Execution log: ['RESERVED', 'CONFIRMED', 'NOTIFIED']
  No rollback needed.


---
## Listings 5.14–5.17: The complete execution pipeline

In [ ]:
# Listing 5.14 — PolicyGate

class PolicyDecision(str, Enum):
    ALLOW            = "allow"
    REQUIRE_APPROVAL = "require_approval"
    BLOCK            = "block"

class PolicyGate:
    """Reads tool metadata and returns an execution decision.
    The model never sees this layer — it operates between selection and execution.
    """

    def check(self, tool: BaseTool) -> PolicyDecision:
        risk = tool.metadata.risk_level
        if risk == RiskLevel.CRITICAL:
            return PolicyDecision.REQUIRE_APPROVAL
        if risk == RiskLevel.HIGH and tool.metadata.requires_confirmation:
            return PolicyDecision.REQUIRE_APPROVAL
        return PolicyDecision.ALLOW


# Test the gate
gate = PolicyGate()
tools_to_test = [
    CustomerLookupTool(),
    IssuedRefundTool(),
    InitiateWireTransferTool(),
]
print("PolicyGate decisions:")
for t in tools_to_test:
    decision = gate.check(t)
    print(f"  {t.metadata.name:<30} [{t.metadata.risk_level:<8}] → {decision}")

PolicyGate decisions:
  get_customer                   [RiskLevel.LOW] → PolicyDecision.ALLOW
  issue_refund                   [RiskLevel.HIGH] → PolicyDecision.REQUIRE_APPROVAL
  initiate_wire_transfer         [RiskLevel.CRITICAL] → PolicyDecision.REQUIRE_APPROVAL


In [ ]:
# Listing 5.15 — IdempotentExecutor (composable, replaces Listing 5.11)
# Listing 5.16 — SafeExecutor (sequences all safeguards)

class IdempotentExecutor:
    """Replaces Listing 5.11: accepts fn callable for composability."""

    def __init__(self, store):
        self.store = store

    async def execute(self, tool_name: str, params: dict, fn) -> ActionResult:
        key = make_idempotency_key(tool_name, params)
        cached = await self.store.get(key)
        if cached:
            print(f"  [IdempotentExecutor] Cache hit for key {key[:12]}...")
            return cached
        result = await fn()
        await self.store.set(key, result)
        return result


class SafeExecutor:
    """Sequences: policy check → input validation → idempotent execution → circuit breaker."""

    def __init__(self, store, circuit_breaker: CircuitBreaker):
        self.gate = PolicyGate()
        self.idempotent = IdempotentExecutor(store)
        self.breaker = circuit_breaker

    async def execute(self, tool: BaseTool, inputs: dict) -> ActionResult:
        # Stage 1: policy gate — no model involvement
        decision = self.gate.check(tool)
        if decision == PolicyDecision.BLOCK:
            return ActionResult(
                status="failure",
                error={"code": "PolicyBlocked", "retryable": False},
            )
        if decision == PolicyDecision.REQUIRE_APPROVAL:
            return ActionResult(
                status="pending",
                error={"code": "AwaitingApproval", "retryable": False},
            )

        # Stage 2: validated execution through circuit breaker
        async def _call():
            validated = tool.metadata.args_schema(**inputs)
            return await tool.execute(validated)

        return await self.idempotent.execute(
            tool_name=tool.metadata.name,
            params=inputs,
            fn=lambda: self.breaker.call(_call),
        )


# Listing 5.17 — ActionEngine
class ActionEngine:
    """Top-level coordinator: retrieval → model selection → safe execution."""

    def __init__(self, registry: ToolRegistry, executor: SafeExecutor):
        self.registry = registry
        self.executor = executor

    def get_candidates(
        self,
        query: str,
        max_risk: RiskLevel = RiskLevel.HIGH,
    ) -> list[BaseTool]:
        """Stage 1: retrieve relevant tools within policy bounds."""
        return self.registry.search(query, top_k=5, max_risk=max_risk)

    async def run(self, tool_name: str, inputs: dict) -> ActionResult:
        """Stage 2: execute the model-selected tool safely."""
        tool = self.registry.get(tool_name)
        if tool is None:
            return ActionResult(
                status="failure",
                error={
                    "code": "ToolNotFound",
                    "message": f"No tool named '{tool_name}'",
                    "retryable": False,
                },
            )
        return await self.executor.execute(tool, inputs)

print("PolicyGate, IdempotentExecutor, SafeExecutor, ActionEngine defined.")

PolicyGate, IdempotentExecutor, SafeExecutor, ActionEngine defined.


---
## Listing 5.18: End-to-end demo (no model)

In [ ]:
# Listing 5.18 — Complete action engine end-to-end demo

class InMemoryStore:
    """In-memory store for demonstration. Replace with Redis or Postgres in production."""
    def __init__(self):
        self._data = {}

    async def get(self, key):
        return self._data.get(key)

    async def set(self, key, value):
        self._data[key] = value


async def demo_listing_518():
    # Build the engine
    reg = ToolRegistry()
    reg.register(CustomerLookupTool())
    reg.register(IssuedRefundTool())
    reg.register(InitiateWireTransferTool())

    store = InMemoryStore()
    breaker = CircuitBreaker(failure_threshold=3, reset_timeout_s=30)
    executor = SafeExecutor(store=store, circuit_breaker=breaker)
    engine = ActionEngine(registry=reg, executor=executor)

    print("=== Stage 1: candidate retrieval (max_risk=MEDIUM) ===")
    candidates = engine.get_candidates(
        query="look up customer account information",
        max_risk=RiskLevel.MEDIUM,
    )
    print(f"  Candidates: {[t.metadata.name for t in candidates]}")

    print("\n=== Stage 2: execute selected tool ===")
    result = await engine.run(
        tool_name="get_customer",
        inputs={"customer_id": "cust_1"},
    )
    print(f"  Execution status : {result.status}")
    print(f"  Domain outcome   : {result.output.status}")
    print(f"  Customer         : {result.output.customer}")

    print("\n=== Idempotency: same call again uses cache ===")
    result2 = await engine.run(
        tool_name="get_customer",
        inputs={"customer_id": "cust_1"},
    )
    print(f"  Status: {result2.status} (from cache)")

    print("\n=== Policy gate: CRITICAL tool requires approval ===")
    result3 = await engine.run(
        tool_name="initiate_wire_transfer",
        inputs={"account_number": "123456", "amount_usd": 1000.0, "recipient_name": "Acme"},
    )
    print(f"  Status: {result3.status} — {result3.error['code']}")

    print("\n=== Tool not found ===")
    result4 = await engine.run(tool_name="nonexistent_tool", inputs={})
    print(f"  Status: {result4.status} — {result4.error['message']}")

await demo_listing_518()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Stage 1: candidate retrieval (max_risk=MEDIUM) ===
  Candidates: ['get_customer']

=== Stage 2: execute selected tool ===
  Execution status : success
  Domain outcome   : found
  Customer         : {'name': 'Asha Gupta', 'email': 'asha@example.com'}

=== Idempotency: same call again uses cache ===
  [IdempotentExecutor] Cache hit for key 10460f690960...
  Status: success (from cache)

=== Policy gate: CRITICAL tool requires approval ===
  Status: pending — AwaitingApproval

=== Tool not found ===
  Status: failure — No tool named 'nonexistent_tool'


---
## Full pipeline with Claude API

This integrates the complete action engine with a live Claude model.  
Claude selects the tool via function calling; the `ActionEngine` executes it safely.

**Flow:**
```
User query
  → ActionEngine.get_candidates()     # registry narrows to relevant tools
  → Build Anthropic tool schemas      # convert BaseTool → Claude tool spec
  → Claude selects tool + arguments   # model reasoning
  → ActionEngine.run()                # policy gate + idempotency + circuit breaker
  → Result fed back to Claude         # final natural language response
```

In [ ]:
# Helper: convert BaseTool metadata → Anthropic tool spec
import json

def tool_to_anthropic_spec(tool: BaseTool) -> dict:
    """Convert a BaseTool to the dict format Claude's API expects."""
    schema = tool.metadata.args_schema.model_json_schema()
    # Remove Pydantic-specific keys that Claude doesn't need
    schema.pop("title", None)
    return {
        "name": tool.metadata.name,
        "description": tool.metadata.description,
        "input_schema": schema,
    }


# Test the converter
spec = tool_to_anthropic_spec(CustomerLookupTool())
print("Anthropic tool spec for CustomerLookupTool:")
print(json.dumps(spec, indent=2))

Anthropic tool spec for CustomerLookupTool:
{
  "name": "get_customer",
  "description": "Retrieve a customer record by unique identifier.",
  "input_schema": {
    "properties": {
      "customer_id": {
        "title": "Customer Id",
        "type": "string"
      }
    },
    "required": [
      "customer_id"
    ],
    "type": "object"
  }
}


In [ ]:
# Full pipeline: Claude selects a tool, ActionEngine executes it

async def run_with_claude(
    user_query: str,
    engine: ActionEngine,
    max_risk: RiskLevel = RiskLevel.MEDIUM,
    verbose: bool = True,
) -> str:
    """
    End-to-end pipeline:
      1. Registry retrieves candidate tools within policy bounds
      2. Claude selects a tool and generates arguments
      3. ActionEngine executes the selected tool safely
      4. Result is fed back to Claude for a natural language response
    """
    # Stage 1: retrieve candidates
    candidates = engine.get_candidates(user_query, max_risk=max_risk)
    if not candidates:
        return "No tools available for this request within the current policy."

    anthropic_tools = [tool_to_anthropic_spec(t) for t in candidates]

    if verbose:
        print(f"Query: {user_query!r}")
        print(f"Candidates passed to Claude: {[t['name'] for t in anthropic_tools]}")

    # Stage 2: Claude selects tool and generates arguments
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        tools=anthropic_tools,
        messages=[{"role": "user", "content": user_query}],
    )

    # Check if Claude made a tool call
    tool_use_block = next(
        (b for b in response.content if b.type == "tool_use"), None
    )

    if tool_use_block is None:
        # Claude answered directly without a tool
        text = next((b.text for b in response.content if b.type == "text"), "")
        if verbose:
            print("Claude answered directly (no tool call).")
        return text

    tool_name = tool_use_block.name
    tool_inputs = tool_use_block.input
    tool_use_id = tool_use_block.id

    if verbose:
        print(f"Claude selected: {tool_name}")
        print(f"Arguments: {json.dumps(tool_inputs, indent=2)}")

    # Stage 3: ActionEngine executes safely
    action_result = await engine.run(tool_name=tool_name, inputs=tool_inputs)

    if verbose:
        print(f"Execution status: {action_result.status}")
        if action_result.output:
            print(f"Domain outcome: {action_result.output.status}")

    # Handle non-success outcomes before sending back to Claude
    if action_result.status == "pending":
        return f"Action '{tool_name}' requires human approval before it can proceed."
    if action_result.status == "failure":
        error_msg = action_result.error.get("message", "Unknown error")
        return f"Action '{tool_name}' failed: {error_msg}"

    # Stage 4: feed result back to Claude for natural language response
    tool_result_content = json.dumps(
        action_result.output.model_dump() if hasattr(action_result.output, 'model_dump')
        else action_result.output
    )

    final_response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        tools=anthropic_tools,
        messages=[
            {"role": "user", "content": user_query},
            {"role": "assistant", "content": response.content},
            {
                "role": "user",
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": tool_use_id,
                    "content": tool_result_content,
                }]
            },
        ],
    )

    final_text = next(
        (b.text for b in final_response.content if b.type == "text"), ""
    )
    return final_text


print("run_with_claude() defined. Run the next cell to test.")

run_with_claude() defined. Run the next cell to test.


In [ ]:
# Build the engine once for all Claude tests
full_registry = ToolRegistry()
full_registry.register(CustomerLookupTool())
full_registry.register(IssuedRefundTool())
full_registry.register(SearchOrdersTool())
full_registry.register(InitiateWireTransferTool())

prod_store = InMemoryStore()
prod_breaker = CircuitBreaker(failure_threshold=3, reset_timeout_s=60)
prod_executor = SafeExecutor(store=prod_store, circuit_breaker=prod_breaker)
prod_engine = ActionEngine(registry=full_registry, executor=prod_executor)

print("Production engine ready with tools:")
for name in full_registry._tools:
    t = full_registry._tools[name]
    print(f"  {name:<35} [{t.metadata.risk_level}]")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Production engine ready with tools:
  get_customer                        [RiskLevel.LOW]
  issue_refund                        [RiskLevel.HIGH]
  search_orders                       [RiskLevel.LOW]
  initiate_wire_transfer              [RiskLevel.CRITICAL]


In [ ]:
# Test 1: Claude looks up a customer — should succeed
print("=" * 60)
print("TEST 1: Customer lookup (LOW risk — auto-executes)")
print("=" * 60)
answer = await run_with_claude(
    user_query="Can you look up the customer with ID cust_1?",
    engine=prod_engine,
    max_risk=RiskLevel.MEDIUM,
)
print(f"\nClaude's response:\n{answer}")

TEST 1: Customer lookup (LOW risk — auto-executes)
Query: 'Can you look up the customer with ID cust_1?'
Candidates passed to Claude: ['get_customer', 'search_orders']
Claude selected: get_customer
Arguments: {
  "customer_id": "cust_1"
}
Execution status: success
Domain outcome: found

Claude's response:
Here are the details for customer **cust_1**:

- **Name:** Asha Gupta
- **Email:** asha@example.com

Is there anything else you'd like to know about this customer?


In [ ]:
# Test 2: Claude tries a CRITICAL tool — policy gate blocks it
print("=" * 60)
print("TEST 2: Wire transfer (CRITICAL — policy gate intercepts)")
print("=" * 60)
answer2 = await run_with_claude(
    user_query="Transfer $500 to account 987654, recipient John Doe.",
    engine=prod_engine,
    max_risk=RiskLevel.CRITICAL,  # allow CRITICAL into candidate set
    verbose=True,
)
print(f"\nClaude's response:\n{answer2}")

TEST 2: Wire transfer (CRITICAL — policy gate intercepts)
Query: 'Transfer $500 to account 987654, recipient John Doe.'
Candidates passed to Claude: ['initiate_wire_transfer', 'issue_refund', 'search_orders', 'get_customer']
Claude selected: initiate_wire_transfer
Arguments: {
  "account_number": "987654",
  "amount_usd": 500,
  "recipient_name": "John Doe"
}
Execution status: pending

Claude's response:
Action 'initiate_wire_transfer' requires human approval before it can proceed.


In [ ]:
# Test 3: Not-found customer — domain outcome vs execution status separation
print("=" * 60)
print("TEST 3: Missing customer — execution succeeds, domain says not_found")
print("=" * 60)
answer3 = await run_with_claude(
    user_query="Look up customer ID cust_999 for me.",
    engine=prod_engine,
    max_risk=RiskLevel.MEDIUM,
)
print(f"\nClaude's response:\n{answer3}")

TEST 3: Missing customer — execution succeeds, domain says not_found
Query: 'Look up customer ID cust_999 for me.'
Candidates passed to Claude: ['get_customer', 'search_orders']
Claude selected: get_customer
Arguments: {
  "customer_id": "cust_999"
}
Execution status: success
Domain outcome: not_found

Claude's response:
It looks like no customer was found for the ID **cust_999**. The record either doesn't exist or may have been removed.

Here are a few things you could try:
- **Double-check the ID** for any typos or formatting issues.
- **Try a different customer ID** if you think the ID may be incorrect.

Let me know how you'd like to proceed!


In [ ]:
# Test 4: Idempotency — same call twice, second hits cache
print("=" * 60)
print("TEST 4: Idempotency — second identical call uses cache")
print("=" * 60)
print("First call:")
r1 = await prod_engine.run("get_customer", {"customer_id": "cust_1"})
print(f"  status={r1.status}  latency={r1.latency_ms:.1f}ms")

print("Second call (identical inputs):")
r2 = await prod_engine.run("get_customer", {"customer_id": "cust_1"})
print(f"  status={r2.status}  latency={r2.latency_ms:.1f}ms  (from cache — latency is 0)")
assert r1.output.customer == r2.output.customer
print("  Results are identical. ✓")

TEST 4: Idempotency — second identical call uses cache
First call:
  [IdempotentExecutor] Cache hit for key 10460f690960...
  status=success  latency=0.1ms
Second call (identical inputs):
  [IdempotentExecutor] Cache hit for key 10460f690960...
  status=success  latency=0.1ms  (from cache — latency is 0)
  Results are identical. ✓


---
## Listings 5.19 & 5.20: Beyond APIs



In [ ]:
# Listing 5.20 — Allowlisted shell execution (safe to run)
import subprocess
import shlex

ALLOWED_COMMANDS = {"ls", "cat", "echo", "pwd", "wc", "head"}

def run_command(command: str, args: list[str]) -> dict:
    if command not in ALLOWED_COMMANDS:
        raise ValueError(f"Command '{command}' is not in the allowlist")
    result = subprocess.run(
        [command] + args,
        capture_output=True,
        text=True,
        timeout=5,
    )
    return {
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }

# Wrap as a BaseTool for use in the action engine
class ShellCommandInput(BaseModel):
    command: str
    args: list[str] = []

class ShellCommandTool(BaseTool):
    """
    Safe shell execution — follows the same BaseTool contract as API tools.
    Registered with HIGH risk and requires_confirmation so it always
    passes through the REQUIRE_APPROVAL policy gate.
    """
    metadata = ToolMetadata(
        name="run_shell_command",
        description=(
            "Execute an allowlisted shell command. "
            "Permitted commands: ls, cat, echo, pwd, wc, head."
        ),
        args_schema=ShellCommandInput,
        risk_level=RiskLevel.HIGH,
        requires_confirmation=True,   # always route to REQUIRE_APPROVAL
        is_idempotent=False,
        timeout_seconds=5.0,
    )

    async def _run(self, input: ShellCommandInput) -> dict:
        return run_command(input.command, input.args)


# Test allowlist enforcement
print("=== Allowlist enforcement ===")
print(run_command("echo", ["hello from the action engine"]))

try:
    run_command("rm", ["-rf", "/tmp/test"])
except ValueError as e:
    print(f"Blocked: {e}")

=== Allowlist enforcement ===
{'returncode': 0, 'stdout': 'hello from the action engine\n', 'stderr': ''}
Blocked: Command 'rm' is not in the allowlist


In [ ]:
# Listing 5.19 — BrowserTool (requires playwright — skip if not installed)
# This cell shows the code; run only if playwright is installed

class BrowserInput(BaseModel):
    url: str

class BrowserTool(BaseTool):
    """
    Web automation tool following the same BaseTool contract.
    Registered as HIGH risk — unstable selectors mean actions
    need human review before deploying to production workflows.
    """
    metadata = ToolMetadata(
        name="extract_web_text",
        description="Load a URL and extract the visible text content of the page body.",
        args_schema=BrowserInput,
        risk_level=RiskLevel.HIGH,
        requires_confirmation=True,
        is_idempotent=True,
        timeout_seconds=30.0,
    )

    async def _run(self, input: BrowserInput) -> dict:
        try:
            from playwright.async_api import async_playwright
            async with async_playwright() as p:
                browser = await p.chromium.launch(headless=True)
                page = await browser.new_page()
                await page.goto(input.url)
                content = await page.inner_text("body")
                await browser.close()
                return {"status": "extracted", "text": content[:500]}  # first 500 chars
        except ImportError:
            return {
                "status": "unavailable",
                "message": "playwright not installed — run: pip install playwright && playwright install"
            }


# Show metadata (does not require playwright to run)
print("BrowserTool metadata:")
print(f"  name              : {BrowserTool.metadata.name}")
print(f"  risk_level        : {BrowserTool.metadata.risk_level}")
print(f"  requires_confirm  : {BrowserTool.metadata.requires_confirmation}")
print(f"  policy decision   : {PolicyGate().check(BrowserTool())}")
print("\nNote: BrowserTool follows the same BaseTool contract as CustomerLookupTool.")
print("It passes through the same policy gate, idempotency, and circuit breaker.")

BrowserTool metadata:
  name              : extract_web_text
  risk_level        : RiskLevel.HIGH
  requires_confirm  : True
  policy decision   : PolicyDecision.REQUIRE_APPROVAL

Note: BrowserTool follows the same BaseTool contract as CustomerLookupTool.
It passes through the same policy gate, idempotency, and circuit breaker.
